# 第2回演习：使用低秩适应（LoRA）学习回答格式，比较训练前后

以虚构课程的常见问题解答（FAQ）为题材，对小型指令微调模型进行短时间的低秩适应（Low-Rank Adaptation，LoRA）训练，比较训练前后**哪些发生变化、哪些没有变化**。评价分为**格式遵守率**（是否遵守包含 `answer` 和 `needs_review` 的 JSON 格式）和**内容正确率**。

**本演习要确认的内容**

1. 即使在提示词中指定格式，小模型也可能无法遵守（第3节，训练前）。
2. 使用 LoRA 训练几分钟后，模型大多能够遵守格式（第5节，训练后）。
3. 即使格式变整齐，也不能保证模型能正确判断 FAQ 未记载的问题：是请求确认，还是直接断言（第5、6节）。
4. 拓展：将同一模型以4位量化加载，即使不训练，输出也可能改变（第8节）。

**流程**

| 章节 | 内容 | 时间参考（T4） |
| --- | --- | --- |
| 0～3 | 环境、配置、生成数据、加载模型、训练前评价 | 约5分钟 |
| 4～7 | 训练、训练后评价、知识问题、比较与提交 | 约10分钟（其中训练4～5分钟） |
| 8（可选） | 以4位量化加载并比较 | 约2分钟 |

课前可以先运行第0～3节，确认环境可用，减少上课当天的问题。Colab 会话断开后变量会消失，因此需要时请从第0节重新执行。

**使用环境**：Google Colab（T4 图形处理器，GPU；免费额度）。模型为 `Qwen/Qwen3.5-0.8B`（Apache-2.0）。数据由本笔记本生成，均为虚构内容，不包含真实截止日期或网址。

**阅读方法**：各节说明包括“操作”“观察重点”和“遇到问题时”。请按顺序从上到下运行单元格；跳过单元格可能导致后续变量不存在。

版本：1.1（2026-10-18）

## 0. 检查环境并安装软件库

**操作**：确认已分配 GPU，并安装已验证的固定版本软件库。

**观察重点**

- 第一个单元格显示 `Tesla T4`。
- 第二个单元格末尾显示版本列表，例如 `transformers: 5.18.0`。安装约需1分钟。

**遇到问题时**

- 出现“未找到 GPU”：在菜单中选择“运行时”→“更改运行时类型”→“T4 GPU”，然后重新执行。资源繁忙无法分配时，稍后再试。
- 与 `cudf`、`pyarrow` 有关的红色 `ERROR: pip's dependency resolver ...` 是 Colab 预装软件包之间的依赖冲突警告；原演习说明称其不影响本实验，可以继续。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 配置

**操作**：把模型名称、训练轮数等集中保存在字典 `CONFIG` 中。第7节会将这些配置写入提交文件，便于之后确认实验条件。

- 固定 `revision` 是为了每次使用相同权重，便于比较训练前后及不同人的结果。
- `num_train_epochs: 20` 表示将14条训练数据重复20轮。每轮2步，共40步；T4训练约4～5分钟。
- `gen: {"do_sample": False}` 表示贪心生成，使训练前后生成条件一致，减少采样波动的影响。

将 `USE_DRIVE` 设为 `True` 会挂载 Google Drive，并把训练后的适配器（adapter）和结果文件保存到云端硬盘，以便会话断开后保留。一次会话内完成时，可以保持 `False`。

**观察重点**：显示的 JSON 配置。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. 生成虚构 FAQ 和监督训练数据

**操作**：根据5条虚构 FAQ，生成训练集（train）、验证集（valid）和测试集（test）。

模型输入是“FAQ＋问题”；期望输出是一行 JSON：`{"answer": <字符串>, "needs_review": <布尔值>}`。

- FAQ 有依据时，依据 FAQ 回答，`needs_review` 为 `false`。
- FAQ 没有记载时，回答“この FAQ には記載がありません。運営に確認が必要です。”（此 FAQ 没有记载，需要向运营方确认），`needs_review` 为 `true`。

**数据划分**

- 将同一 FAQ 条目的不同问法按**模板**分配到 train / valid / test，避免训练中见过的原句直接出现在测试中。
- test 有21题（有依据15题、无依据6题），其中另行准备了只用于 test 的问题。
- 标准答案保存在 `gold` 中，包括 `needs_review` 的值和回答必须包含的词句；不传给模型。
- train 有14条（有依据10条、无依据4条），valid 有7条。FAQ 只有5项，数据量很少，同一例子会重复训练。重复次数由第4节的 `num_train_epochs` 决定，而不是复制数据；数据文件保存实际样本，训练量由配置控制。

**观察重点**：显示 `train=14 valid=7 test=21 gold=21` 和一条训练示例。如果同一个问题在 train / valid / test 中重复，会在这里报错。

In [ ]:
FAQ = {
    "deadline": "課題の提出期限は 9 月 30 日 23:59（日本時間）です。",
    "destination": "提出先は学習ポータルの「課題」欄です。",
    "format": "提出形式は PDF 1 ファイルで、ファイル名は 学籍番号_課題番号.pdf です。",
    "office_hours": "質問は毎週火曜 17:00〜18:00 のオフィスアワーで受け付けます。",
    "late": "期限後の提出は、期限から 48 時間以内なら減点のうえ受け付けます。",
}
FAQ_TEXT = "\n".join(f"- {v}" for v in FAQ.values())

# 根拠あり: (FAQ キー, 質問テンプレート群, 正解の answer に含まれるべき語)
GROUNDED = {
    "deadline": (["課題の提出期限はいつですか", "いつまでに出せばいいですか", "締切を教えてください", "提出の期限は？"], ["9 月 30 日", "23:59"]),
    "destination": (["提出先はどこですか", "どこにアップロードしますか", "課題はどこから出しますか", "提出場所は？"], ["学習ポータル", "課題"]),
    "format": (["提出形式を教えてください", "ファイル名の決まりはありますか", "何のファイル形式で出しますか", "PDF 以外でも出せますか"], ["PDF"]),
    "office_hours": (["質問はいつできますか", "オフィスアワーはいつですか", "先生に質問する時間はありますか", "相談できる時間帯は？"], ["火曜", "17:00"]),
    "late": (["遅れて出したらどうなりますか", "期限を過ぎた場合は受け付けますか", "遅延提出のルールは？", "締切後の扱いを教えてください"], ["48 時間", "減点"]),
}
# 根拠なし: FAQ に無いので needs_review=true が正解
UNGROUNDED = [
    "再提出はできますか", "欠席した回の扱いを教えてください", "他の受講者の点数は見られますか",
    "成績はいつ出ますか", "提出期限は延長できますか", "グループでの提出は可能ですか",
    "課題の配点はどうなっていますか", "講義の録画は公開されますか",
]

# test 専用の追加質問（train / valid には入れない）
TEST_EXTRA_GROUNDED = {
    "deadline": ["提出期限の日付と時刻を教えてください", "課題の締切は何月何日の何時ですか"],
    "destination": ["課題の提出はどこで行いますか", "提出物を出す先を教えてください"],
    "format": ["提出するファイルの形式は何ですか", "提出ファイルは何形式で、いくつ出しますか"],
    "office_hours": ["オフィスアワーの曜日と時間を教えてください", "質問を受け付けてもらえるのは何曜日の何時からですか"],
    "late": ["提出が締切に間に合わなかったらどうなりますか", "期限後に提出した課題は受理されますか"],
}
TEST_EXTRA_UNGROUNDED = [
    "課題の文字数に制限はありますか", "提出した課題へのフィードバックはもらえますか",
    "参考文献の書き方に指定はありますか", "オフィスアワーは事前予約が必要ですか",
]
UNGROUNDED_ANSWER = "この FAQ には記載がありません。運営に確認が必要です。"

def make_example(question: str, answer: str, needs_review: bool) -> dict:
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"question": question, "faq": FAQ_TEXT, "target": target, "needs_review": needs_review}

def split_templates(templates):
    # テンプレートの先頭 2 つを train、3 つ目を valid、4 つ目を test に固定（言い換えを跨がせない）
    return templates[:2], templates[2:3], templates[3:4]

train, valid, test, gold = [], [], [], {}
for key, (templates, must_include) in GROUNDED.items():
    tr, va, te = split_templates(templates)
    for q in tr: train.append(make_example(q, FAQ[key], False))
    for q in va: valid.append(make_example(q, FAQ[key], False))
    for q in te + TEST_EXTRA_GROUNDED[key]:
        test.append(make_example(q, FAQ[key], False)); gold[q] = {"needs_review": False, "must_include": must_include}
for i, q in enumerate(UNGROUNDED):
    ex = make_example(q, UNGROUNDED_ANSWER, True)
    bucket = [train, train, valid, test][i % 4]   # 根拠なしの質問は 4 問ごとに train 2 / valid 1 / test 1 に振る
    bucket.append(ex)
    if bucket is test: gold[q] = {"needs_review": True, "must_include": ["確認"]}
for q in TEST_EXTRA_UNGROUNDED:
    test.append(make_example(q, UNGROUNDED_ANSWER, True)); gold[q] = {"needs_review": True, "must_include": ["確認"]}

# train は 14 件（根拠あり 10、根拠なし 4）。同じ例を何回学習するかは、データを複製せず、セクション 4 の num_train_epochs で決める
all_questions = [r["question"] for r in train + valid + test]
assert len(set(all_questions)) == len(all_questions), "質問が train / valid / test のどこかで重複しています。テンプレートを見直してください"
print(f"train={len(train)} valid={len(valid)} test={len(test)} gold={len(gold)}")
print("学習例の 1 件 | 質問:", train[0]["question"], "| 望ましい出力:", train[0]["target"])
_ = (WORK / "train.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in train), encoding="utf-8")
_ = (WORK / "test.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in test), encoding="utf-8")
_ = (WORK / "gold.json").write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")

## 3. 加载模型并进行训练前评价

**操作**：不使用量化，以32位精度加载基础模型，对 test 的21题生成回答并评分。

提示词 `SYSTEM` 已经要求模型“必须以一行 JSON 返回 `answer` 和 `needs_review`”。也就是说，**已经给出格式指令**；训练前要检查模型能否遵守。

**评分方式**（第5节使用相同函数）

- **格式**：能解析为 JSON，恰好包含 `answer` 和 `needs_review` 两个键，类型分别为字符串和布尔值。
- **内容**：`needs_review` 与标准答案一致，且 `answer` 包含规定的关键词，例如“9 月 30 日”和“23:59”。

内容评分需要 `needs_review` 一致。因此，缺少该键时，即使文字正确，内容也判为不通过。请自行阅读表格中的回答原文。

`Qwen/Qwen3.5-0.8B` 也能处理图像，但通过 `AutoModelForCausalLM` 加载时只使用文本部分。

**观察重点**

- 第一个单元格可能显示 `重みのメモリ(GB): 3.01`，意为权重占用约3.01 GB。下载模型约1.75 GB，约需1分钟。
- 第二个单元格显示一行汇总（`format_rate`、`content_rate` 等）及21题的输出表；评价约需1分钟。训练前许多输出可能缺少 `needs_review`。

**原演习说明中可忽略的提示**

- `You are sending unauthenticated requests to the HF Hub`：未登录 Hugging Face 下载模型的提示。公开模型可不登录下载；需要登录的情况包括受限或私有模型、上传内容，以及请求达到未登录额度。原资料引用的未登录上限为每个 IP 每5分钟3000次文件获取（2025年9月时点；[Hub 限流说明](https://huggingface.co/docs/hub/rate-limits)）。本笔记本所用模型公开且不要求同意条款；如需登录，可在 Colab 左侧密钥图标中登记名为 `HF_TOKEN` 的秘密，并允许本笔记本访问。
- `... is falling back to its reference PyTorch implementation ...`：部分线性注意力层未安装高速实现，程序会使用较慢的参考实现，提示约出现4行。

**此处检查**：阅读缺失的键和回答与 FAQ 的一致性。第5节会与训练后输出对照。原文也保存在 `/content/qlora_lab/eval_before.json`。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

# 形式はプロンプトで指示している。モデルがこれを守れるかが学習前の確認点
SYSTEM = "あなたは講座 FAQ の回答係です。与えられた FAQ だけを根拠に、必ず JSON 1 行で答えてください。形式: {\"answer\": 文字列, \"needs_review\": 真偽値}。FAQ に記載が無いことは answer に『確認が必要』と書き、needs_review を true にします。"

def build_messages(example):
    user = f"FAQ:\n{example['faq']}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example):
    # enable_thinking=False: 思考ブロックを空にして回答だけを生成させる（モデルを替えても条件が揃うように明示）
    prompt = tok.apply_chat_template(build_messages(example), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)   # 出力の原文を省略せずに表示する

def check_format(text: str):
    '''JSON として読め、必須キーと型が合うか。失敗理由も返す'''
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    '''内容の合否: needs_review が正解と一致し、必須語句を answer に含む'''
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str):
    '''test の 21 問を生成して採点し、要約と質問ごとの結果を返す。原文は eval_<label>.json に保存する'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex)
        ok_fmt, reason, obj = check_format(raw)
        # 形式が壊れていても JSON らしい部分が読めるなら、内容だけは別に採点する
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "raw": raw, "format_ok": ok_fmt, "format_reason": reason,
                     "content_ok": ok_content, "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    summary = {"label": label, "n": n,
               "format_rate": sum(r["format_ok"] for r in rows)/n,
               "content_rate": sum(r["content_ok"] for r in rows)/n,
               "both_rate": sum(r["both_ok"] for r in rows)/n,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
# 質問ごとの出力。「正解の needs_review」と見比べ、どのキーが欠けているか、文面は FAQ と合っているかを読む
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "学習前の出力": r["raw"],
                       "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"], "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 训练（T4约4～5分钟）

### LoRA 与 QLoRA 的区别

- **LoRA**：固定基础模型权重，只训练加在部分层上的小矩阵，即适配器（adapter）。
- **QLoRA**：将基础权重量化为4位保存，再训练适配器；适配器本身不量化。

QLoRA 的作用是减少基础权重的存储。相较16位，4位权重理论存储约为四分之一；70亿参数的权重约从14 GB降到3.5 GB，不含其他开销。

| 方面 | LoRA（不量化） | QLoRA（4位） |
| --- | --- | --- |
| 权重内存 | 较大 | 较小，约为16位的1/4 |
| 基础模型输出 | 保持原精度 | 可能受量化误差影响，程度因模型而异 |
| 训练速度 | 基准 | 不一定更快，量化权重转换可能使其变慢 |
| 依赖 | peft | peft 与 bitsandbytes，需要 CUDA GPU |

**实现 QLoRA 时的注意事项**

- 加载时传入 `BitsAndBytesConfig(load_in_4bit=True, ...)`，添加 LoRA 前调用 `prepare_model_for_kbit_training(model)`。
- 存储精度（4位）与计算精度（`bnb_4bit_compute_dtype`，例如16位）是不同设置。
- 训练前后要使用相同的量化条件，否则会混淆量化与训练的影响。
- 保存或复用适配器时，同时记录基础模型 ID、revision 和量化设置。
- 有些模型不建议以4位训练。[相关训练库指南](https://unsloth.ai/docs/models/qwen3.5/fine-tune)指出 Qwen3.5 的量化误差可能较大，因此不推荐4位训练。

本实验的模型不量化也能放入 T4，因此使用 LoRA。第8节可观察4位加载后的内存和输出变化。

### 执行训练

**操作**：把14条训练数据转换成对话格式（system / user / assistant），添加 LoRA 后训练20轮、40步；适配器保存在 `adapter/`。

- `target_modules` 指定插入 LoRA 的层，包括注意力投影（例如 `q_proj`）和多层感知机投影（例如 `gate_proj`）。本模型24层中18层使用线性注意力，只有其余6层包含 `q_proj` 等；所有层都有多层感知机。
- 批次4 × 梯度累积2，每步相当于处理最多8条。14条数据每轮2步（8条和6条），20轮共40步，每条样本使用20次。学习率预热和余弦衰减作用于这40步。
- 训练后的模型保存在 `tuned`。由于适配器插入基础模型，训练后即使用 `model` 生成也会包含适配器效果。需要训练前输出时，按第6节暂时禁用适配器。

**观察重点**

- `trainable params: ... || all params: ... || trainable%: ...`：LoRA 只训练全部参数中的一小部分。
- `Training Loss` 表每10步记录一次；原资料参考从约1.5下降到第40步约0.01～0.02。若没有下降，检查配置。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`：原资料给出的 T4 参考时间为240～290秒。步数和轮数用于确认14条数据重复20轮的执行情况。

`The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` 表示结束等词元配置已与分词器对齐。

训练中断或无法训练时，请阅读第4b节。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def to_chat(example):
    # 学習データは「入力（system + user）」と「望ましい出力（assistant）」の対。評価と同じ build_messages を使う
    return {"messages": build_messages(example) + [{"role": "assistant", "content": example["target"]}]}

train_ds = Dataset.from_list([to_chat(e) for e in train])
valid_ds = Dataset.from_list([to_chat(e) for e in valid])

model.config.use_cache = False
# Qwen3.5 は 24 層のうち 18 層が線形注意で、q/k/v/o_proj を持つのは残りの 6 層。gate/up/down_proj（MLP）は全層にある
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
USE_AMP = False  # 通常は変えない。True にすると計算だけ 16bit の混合精度になる（CUDA のみ。重みは 32bit のまま）。速くなる一方、損失が nan になる場合がある
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_steps=10, save_total_limit=2,
                fp16=USE_AMP, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, eval_dataset=valid_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()   # LoRA で学習するパラメータの割合
t0 = time.time()
trainer.train()  # 途中で止まった場合の再開は 4b を参照
print(f"学習時間: {time.time()-t0:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()  # 学習モードのままだと dropout が効き、評価の出力が揺れる（代入はモデル全体の表示を抑えるため）

### 4b. 训练中断或无法训练时

**中途停止**：检查点每10步保存在 `/content/qlora_lab/ckpt/`。如果文件仍在，重新运行第0～3节，再把训练单元格的 `trainer.train()` 改为 `trainer.train(resume_from_checkpoint=True)` 并执行，即可从最后检查点继续到40步、20轮。如果运行时被删除、文件不存在，则需要重新训练或采用下面的方法。

**无法训练**：例如没有 GPU 或规定时间内无法完成时，可使用讲师在 T4 上运行同一笔记本得到的输出与分数进行比较。第5节起需要训练后的模型 `tuned` 才能运行，因此请对照讲师结果阅读说明。提交时将第7节的 `executed_training_myself` 设为 `False`，并写明自己实际完成的范围。

## 5. 训练后评价（测试题与生成设置和训练前相同）

**操作**：使用训练后的 `tuned`，以完全相同的21题和生成设置进行评价，对照训练前后。改变的是是否使用适配器。

**观察重点**

- 比较表中 `format_rate`（格式遵守率）和 `content_rate`（内容正确率）的变化。
- `退行`：训练前格式和内容都通过、训练后却失败的问题；理想情况下为空列表。
- 阅读每题原文，区分剩余错误：无依据问题套用 FAQ 其他内容并断言；有依据问题却声称没有记载并要求确认。

训练后的回答可能接近训练范例中的 FAQ 原文。格式和措辞变整齐，并不保证模型已经学会判断 FAQ 是否有依据。

即使固定随机种子，GPU 训练也不保证完全复现。比例相近时，具体失败问题也可能不同；请按错误类型比较。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
# 学習前は形式も内容も通ったのに、学習後に落ちた質問（退行）
regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

# 質問ごとの出力原文。形式（JSON・必須キー・型）と内容（needs_review の一致と必須語句）の判定を自分の目で確かめる
detail = pd.DataFrame([{"質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 考查知识的问题（学会格式不保证知识正确）

**操作**：在格式实验之外，询问3个 FAQ 和训练数据都没有涉及的事实，并加入1个 FAQ 内的问题作为对照。对无依据问题，期望回答“需要确认（`needs_review: true`）”。

| 分类 | 含义 |
| --- | --- |
| `format_error` | 不遵守 JSON 格式，训练前较常见 |
| `needs_review` | 回答“没有记载，需要确认”，对无依据问题是期望结果 |
| `asserted` | 以 `needs_review: false` 作出断言；若 FAQ 无依据，则属于无依据断言 |

**观察重点**：训练前可能多为格式错误，训练后可能同时出现请求确认与断言。表格也显示原文，请检查断言时是否挪用了 FAQ 其他条目。不要预先断定“微调绝对不能改善知识”，应报告本次数据下的实际结果。

训练前输出通过临时禁用适配器取得：`tuned.disable_adapter()`。

In [ ]:
KNOWLEDGE_QUESTIONS = [
    "この講座の最終課題の締切はいつですか",       # FAQ に無い（架空）
    "担当教員の名前は何ですか",                     # FAQ に無い
    "次回の講義のテーマは何ですか",                 # FAQ に無い
    "課題の提出期限はいつですか",                   # FAQ にある（対照）
]
def classify(raw):
    ok, _, obj = check_format(raw)
    if not ok or obj is None: return "format_error"
    return "needs_review" if obj["needs_review"] else "asserted"

knowledge_rows = []
for q in KNOWLEDGE_QUESTIONS:
    ex = make_example(q, "", True)
    # LoRA は基盤モデルの層に差し込まれるので、学習後は `model` で生成しても adapter 込みになる。学習前の出力は adapter を一時的に無効にして取る
    with tuned.disable_adapter():
        before_raw = generate(tuned, ex)
    after_raw = generate(tuned, ex)
    knowledge_rows.append({"質問": q, "FAQ にある": q in gold, "学習前の分類": classify(before_raw), "学習前の出力": before_raw,
                 "学習後の分類": classify(after_raw), "学習後の出力": after_raw})
display(pd.DataFrame(knowledge_rows))

## 7. 比较表与提交

**操作**：将配置、软件库版本、训练前后汇总和退化问题写入 `result.json`。

**提交内容**

- `result.json`、`eval_before.json`、`eval_after.json`：位于 `/content/qlora_lab`，可在左侧文件列表右键下载；`USE_DRIVE = True` 时也会复制到 Drive。
- 第5节的比较表及剩余错误的类型说明。
- 第6节的结果：哪些回答变为请求确认，哪些仍然断言。
- 自己实际执行训练的范围；使用讲师结果时，将 `executed_training_myself` 设为 `False`。

In [ ]:
result = {
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "before": before_summary, "after": after_summary,
    "regressed": regressed,
    "knowledge": knowledge_rows,
    "executed_training_myself": True,   # 講師の結果を使った場合は False にする
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({k: v for k, v in result.items() if k != "knowledge"}, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "test.jsonl", "gold.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

## 8. 拓展：以4位量化加载，比较内存和输出（可选）

**操作**：以4位（NF4）加载同一模型，观察权重内存和训练前输出的变化。QLoRA 是在4位基础模型上加入 LoRA 进行训练的方法；本节不训练。

- bitsandbytes 需要 CUDA GPU，只能在 GPU 运行时使用，安装约1分钟。
- 4位加载使用 `device_map="auto"` 分配到 GPU，以适配 bitsandbytes 的加载方式。
- 如果尝试对4位模型训练，先执行 `prepare_model_for_kbit_training(model_4bit)`，再按照第4节训练，并确保训练前后都以4位基础模型评价。

**观察重点**

- 原资料参考：32位权重约3 GB，4位不足1 GB。
- 因为没有训练，与32位训练前结果 `before` 的差异来自量化。有些问题可能变好；有些可能把 `needs_review` 输出成字符串 `"false"` 而未通过类型检查，或对无依据问题直接断言“是”“没有”。变化方向并不固定。

In [ ]:
%pip install -q "bitsandbytes==0.50.*"
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",   # 保持精度: 4bit（NF4）
                         bnb_4bit_compute_dtype=torch.float16,             # 計算精度: 16bit（T4 は bf16 に対応していない）
                         bnb_4bit_use_double_quant=True)
model_4bit = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], quantization_config=bnb, device_map="auto")
print("重みのメモリ(GB) | 量子化なし（32bit）:", BASE_MEMORY_GB, "| 4bit:", round(model_4bit.get_memory_footprint() / 1e9, 2))

# 学習はしていないので、量子化なしの学習前（before）との差が量子化の影響
q_summary, q_rows = evaluate(model_4bit, "before_4bit")
display(pd.DataFrame([before_summary, q_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"質問": r["question"], "正解の needs_review": r["gold_needs_review"], "4bit・学習前の出力": r["raw"],
                       "形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}", "形式の失敗理由": r["format_reason"]} for r in q_rows]))